# LifeLedger — Phase 7 · Generational Engine Validation

Validates:
1. Career salary curves (all 10 paths, UK + US)
2. UK and US tax calculations
3. University cost models (Plan 5 loan + 529 plan)
4. Offspring projection (FIRE detection, account growth)
5. Wealth transfer / IHT / US estate tax
6. Country comparison (UK vs US parent paths)
7. **Validation figures** from spec:
   - UK retirement wealth ~£5.5M
   - US retirement wealth ~$12.9M
   - Estate to offspring: £24M (UK) / $53M (US)
8. 4-panel dashboard chart

In [ ]:
import sys, logging
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings; warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

from backend.engine.generational_engine import (
    GenerationalEngine, salary_at_age, calculate_uk_tax,
    calculate_us_tax, calculate_uk_university_cost,
    calculate_us_university_cost, UniversityConfig,
    calculate_wealth_transfer, EstateConfig,
    load_generational_config,
)
from backend.engine.country_comparison_engine import (
    CountryComparisonEngine, build_uk_path_config, build_us_path_config,
    GenerationalMacro,
)

CFG_PATH = ROOT / 'config' / 'generational' / 'generational_config.yaml'
engine = GenerationalEngine(load_generational_config(str(CFG_PATH)))
print(f'Career paths: {list(engine._career_paths.keys())}')
print(f'Offspring:    {[o.name for o in engine._offspring]}')

## 1 · Salary Curves

In [ ]:
swe = engine._career_paths['software_engineer']
for age in [22, 30, 40, 48, 60]:
    uk_s = salary_at_age(swe.uk, age)
    us_s = salary_at_age(swe.us, age)
    print(f'Age {age}: UK=£{uk_s:,.0f}  US=${us_s:,.0f}  ratio={us_s/uk_s:.1f}×')

assert salary_at_age(swe.uk, 21) == 0, 'Before entry_age should be 0'
assert salary_at_age(swe.uk, 22) == swe.uk.entry_salary
assert salary_at_age(swe.uk, 48) == swe.uk.peak_salary
print('\n✅ Salary curve assertions passed')

## 2 · Tax Calculations

In [ ]:
print('UK tax at various salaries:')
for gross in [30_000, 60_000, 100_000, 150_000]:
    tax, ni = calculate_uk_tax(gross)
    net = gross - tax - ni
    eff = (tax + ni) / gross
    print(f'  £{gross:>7,.0f} → tax=£{tax:>7,.0f}  NI=£{ni:>6,.0f}  net=£{net:>7,.0f}  eff={eff:.1%}')

assert calculate_uk_tax(12570)[0] == 0, 'No tax at personal allowance'
assert calculate_uk_tax(50270)[0] > 0

print('\nUS federal tax (single filer):')
for gross in [100_000, 200_000, 400_000, 600_000]:
    fed, fica, st = calculate_us_tax(gross, pretax_401k=23_500, state_rate=0.0)
    net = gross - fed - fica - 23_500
    eff = (fed + fica) / gross
    print(f'  ${gross:>7,.0f} → fed=${fed:>7,.0f}  fica=${fica:>5,.0f}  net=${net:>7,.0f}  eff={eff:.1%}')

assert calculate_us_tax(0)[0] == 0
print('\n✅ Tax assertions passed')

## 3 · University Costs

In [ ]:
uni_cfg = engine._uni_cfg

uk_uni = calculate_uk_university_cost(uni_cfg, duration=3)
print('UK Plan 5 (3yr):')
print(f'  Total tuition:   £{uk_uni.total_tuition:,.0f}')
print(f'  Total living:    £{uk_uni.total_living:,.0f}')
print(f'  Parental outlay: £{uk_uni.parental_outlay:,.0f}')
print(f'  Loan at grad:    £{uk_uni.loan_balance_at_graduation:,.0f}')
print(f'  Repayment yrs:   {uk_uni.projected_loan_repayment_years:.0f}')
print(f'  Written off:     {uk_uni.projected_loan_write_off}')

us_uni = calculate_us_university_cost(uni_cfg, duration=4)
print('\nUS 529 Plan (4yr, mid scenario):')
print(f'  Total tuition:   ${us_uni.total_tuition:,.0f}')
print(f'  Total living:    ${us_uni.total_living:,.0f}')
print(f'  Parental outlay: ${us_uni.parental_outlay:,.0f}')
print(f'  529 used:        ${us_uni.loan_taken:,.0f}')

# Spec targets: UK £98k parental outlay, US $134k
print(f'\nSpec: UK ~£98k, US ~$134k')
print(f'Got:  UK £{uk_uni.total_tuition + uk_uni.total_living:.0f} total (£{uk_uni.parental_outlay:.0f} parental)')
print(f'Got:  US ${us_uni.total_tuition + us_uni.total_living:.0f} total (${us_uni.parental_outlay:.0f} parental)')

assert uk_uni.total_tuition > 0
assert us_uni.loan_taken > 0
print('\n✅ University assertions passed')

## 4 · Offspring Projection (UK path)

In [ ]:
from backend.engine.generational_engine import GenerationalMacro

cfg_dict = load_generational_config(str(CFG_PATH))
raw_uk_mid = cfg_dict.get('generational',{}).get('country_macro',{}).get('UK',{}).get('mid',{})
uk_macro = GenerationalMacro(
    inflation=float(raw_uk_mid.get('inflation',0.025)),
    equity_real_return=float(raw_uk_mid.get('equity_real_return',0.05)),
    salary_real_growth=float(raw_uk_mid.get('salary_real_growth',0.01)),
    healthcare_annual=0.0,
)

from backend.engine.generational_engine import OffspringProjectionEngine
oproj_engine = OffspringProjectionEngine(engine._career_paths, engine._uni_cfg)

offspring_cfg = engine._offspring[0]
proj = oproj_engine.project(offspring_cfg, uk_macro, career_path_id='software_engineer', country='uk')

print(f'Name:            {proj.name}')
print(f'Career:          {proj.career_path} / {proj.country}')
print(f'FIRE year:       {proj.fire_year}')
print(f'FIRE age:        {proj.fire_age}')
print(f'Peak net worth:  £{proj.peak_net_worth:,.0f} ({proj.peak_net_worth_year})')
print(f'Lifetime tax:    £{proj.lifetime_tax:,.0f}')
print(f'Lifetime earn:   £{proj.lifetime_earnings:,.0f}')
print(f'UK uni loan:     £{proj.university_cost.loan_balance_at_graduation:,.0f}')

# Show key years
print('\nWealth at key years:')
for yr in [2040, 2050, 2060, 2070, 2080, 2090]:
    s = proj.year(yr)
    if s: print(f'  {yr} (age {s.age}): £{s.total_net_worth:,.0f}  [{s.career_phase}]')

assert proj.lifetime_earnings > 0
assert any(s.fire_achieved for s in proj.years), 'Offspring should achieve FIRE in UK/SWE career'
print('\n✅ Offspring projection assertions passed')

## 5 · Wealth Transfer / IHT / US Estate Tax

In [ ]:
# Test with spec validation figures
transfer_uk = calculate_wealth_transfer(
    parent_wealth_gbp=5_500_000,
    pension_value_gbp=623_000,
    property_value_gbp=800_000,
    mortgage_balance_gbp=0,
    death_year=2070,
    estate_cfg=engine._estate_cfg,
    fx_rate=1.27,
    has_surviving_partner=True,
)

print('UK path estate transfer (£5.5M portfolio):')
print(f'  Gross estate:       £{transfer_uk.gross_estate_gbp:,.0f}')
print(f'  Pension excluded:   £{transfer_uk.pension_outside_gbp:,.0f}')
print(f'  IHT liability:      £{transfer_uk.iht_liability_gbp:,.0f}')
print(f'  Net to offspring:   £{transfer_uk.net_to_offspring_gbp:,.0f}')
print(f'  Spec target:        ~£24M')

print()
print('US path estate transfer ($12.9M at 1.27 FX):')
print(f'  US estate tax:      ${transfer_uk.us_estate_tax_usd:,.0f}')
print(f'  Net (USD):          ${transfer_uk.net_to_offspring_usd:,.0f}')
print(f'  Spec target:        ~$53M (at peak US wealth)')

assert transfer_uk.iht_liability_gbp >= 0
assert transfer_uk.net_to_offspring_gbp <= transfer_uk.gross_estate_gbp
print('\n✅ Wealth transfer assertions passed')

## 6 · Country Comparison (UK vs US parent paths)

In [ ]:
raw_mac = cfg_dict.get('generational',{}).get('country_macro',{})
def get_macro(country_key, scenario='mid'):
    r = raw_mac.get(country_key,{}).get(scenario,{})
    return GenerationalMacro(
        inflation=float(r.get('inflation',0.025)),
        equity_real_return=float(r.get('equity_real_return',0.05)),
        salary_real_growth=float(r.get('salary_real_growth',0.01)),
        healthcare_annual=float(r.get('healthcare_working', r.get('annual_healthcare_cost',0))),
        healthcare_aca_bridge=float(r.get('healthcare_aca_bridge',0)),
        healthcare_medicare=float(r.get('healthcare_medicare',0)),
        healthcare_late_life=float(r.get('healthcare_late_life',0)),
    )

from backend.engine.country_comparison_engine import CountryComparisonEngine, build_uk_path_config, build_us_path_config
comp = CountryComparisonEngine(get_macro('UK'), get_macro('US'))
uk_path_cfg = build_uk_path_config(cfg_dict, fx_rate=1.27)
us_path_cfg = build_us_path_config(cfg_dict, fx_rate=1.27)
result = comp.compare(uk_path_cfg, us_path_cfg, birth_year_primary=1980)

print('Country comparison (mid scenario, FX=1.27):')
print(f'UK retire wealth:  £{result.uk_path.wealth_at_retirement:,.0f} (target ~£5.5M)')
print(f'US retire wealth:  ${result.us_path.wealth_at_retirement:,.0f} (target ~$12.9M)')
print(f'US advantage:      £{result.us_advantage_at_retirement_gbp:,.0f} GBP')
print(f'Break-even year:   {result.break_even.break_even_year}')
print(f'Lifetime tax delta (US-UK): £{result.lifetime_tax_delta_gbp:,.0f}')
print(f'Lifetime HC delta (US-UK):  £{result.lifetime_healthcare_delta_gbp:,.0f}')
print(f'UK estate (net IHT): £{result.uk_estate_gbp:,.0f}')
print(f'US estate (net tax): £{result.us_estate_gbp:,.0f}')

print('\nKey ages:')
for k in result.key_ages:
    delta_sign = '+' if k.delta_gbp >= 0 else ''
    print(f'  Age {k.age:2d} ({k.year}): UK=£{k.uk_wealth_gbp/1e6:.2f}M  US=£{k.us_wealth_gbp/1e6:.2f}M  Δ={delta_sign}£{k.delta_gbp/1e6:.2f}M')

assert result.uk_path.wealth_at_retirement > 0
assert result.us_path.wealth_at_retirement > 0
print('\n✅ Country comparison assertions passed')

## 7 · Phase 7 Dashboard Chart

In [ ]:
import numpy as np

fig = plt.figure(figsize=(16, 12), facecolor='#0d1117')
gs  = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e', labelsize=8)
    ax.spines[:].set_color('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.5)

fig.suptitle('LifeLedger Phase 7 — Generational Planning Dashboard', color='#e6edf3', fontsize=13, y=0.98)

# ── Panel 1: UK vs US parent wealth trajectory ────────────────────────────────
uk_years = [s.year for s in result.uk_path.years]
uk_wealth = [s.total_wealth_gbp/1e6 for s in result.uk_path.years]
us_wealth = [s.total_wealth/1.27/1e6 for s in result.us_path.years]

ax1.plot(uk_years, uk_wealth, color='#0e9aad', linewidth=2, label='UK Path (GBP)')
ax1.plot(uk_years[:len(us_wealth)], us_wealth, color='#d4a843', linewidth=2, label='US Path (GBP equiv)')
if result.break_even.break_even_year:
    ax1.axvline(result.break_even.break_even_year, color='#2dbd7e', linewidth=1.5,
                linestyle='--', label=f'Break-even {result.break_even.break_even_year}')
ax1.axvline(result.uk_path.retire_year, color='#8b949e', linewidth=1, linestyle=':', alpha=0.7)
ax1.set_title('Parent Wealth — UK vs US Path (£M)', color='#e6edf3', fontsize=10)
ax1.set_ylabel('Net Worth (£M)', color='#8b949e')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:.1f}M'))
ax1.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)

# ── Panel 2: Offspring wealth trajectory UK career paths ─────────────────────
for i, career_id in enumerate(list(engine._career_paths.keys())[:5]):
    proj2 = oproj_engine.project(offspring_cfg, uk_macro, career_path_id=career_id, country='uk')
    off_years = [s.year for s in proj2.years if s.year >= 2035]
    off_nw    = [s.total_net_worth/1e6 for s in proj2.years if s.year >= 2035]
    col = ['#0e9aad','#d4a843','#2dbd7e','#a78bfa','#f97316'][i]
    ax2.plot(off_years, off_nw, color=col, linewidth=1.5,
             label=engine._career_paths[career_id].label)
ax2.set_title('Offspring Wealth (UK, 5 careers)', color='#e6edf3', fontsize=10)
ax2.set_ylabel('Net Worth (£M)', color='#8b949e')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:.1f}M'))
ax2.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=7, ncol=1)

# ── Panel 3: Estate comparison UK vs US at key ages ───────────────────────────
ages = [k.age for k in result.key_ages]
uk_w = [k.uk_wealth_gbp/1e6 for k in result.key_ages]
us_w = [k.us_wealth_gbp/1e6 for k in result.key_ages]
x = np.arange(len(ages))
w = 0.35
ax3.bar(x - w/2, uk_w, w, color='#0e9aad', alpha=0.85, label='UK')
ax3.bar(x + w/2, us_w, w, color='#d4a843', alpha=0.85, label='US (GBP)')
ax3.set_xticks(x)
ax3.set_xticklabels([f'Age {a}' for a in ages], fontsize=8, color='#8b949e')
ax3.set_title('Wealth at Key Ages (£M)', color='#e6edf3', fontsize=10)
ax3.set_ylabel('£M', color='#8b949e')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:.1f}M'))
ax3.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8)

plt.savefig('phase7_generational_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Phase 7 dashboard saved.')

## ✅ Phase 7 Validation Complete